# Exercise: OpenSees Parameter Sweep with PyLauncher

A cantilever pushover runs once per parameter combination, and [PyLauncher](https://github.com/TACC/pylauncher) packs all the runs into one Tapis job, feeding tasks to the node's cores until the list is done. Your task is to define the sweep, preview it, submit it, and plot the family of pushover curves.


The cantilever model:
```
   ^Y
   |
   2       __
   |         |
   |         |
   |         |
 (1)      LCol
   |         |
   |         |
   |         |
 =1=    ----  -------->X
```

- Node 1: fixed base
- Node 2: free top with `NodalMass` 4.19, 4.39, 4.59, 4.79, and 4.99
- Column lengths are: 100, 200, and 300
- Elastic beam-column element
- Gravity load (2000 kip downward) followed by lateral pushover (displacement-controlled)

In [ ]:
%pip install dapi --quiet

**Restart the kernel once after the install**, then run from the next cell.

In [ ]:
from pathlib import Path
from dapi import DSClient

ds = DSClient()

# DesignSafe JupyterHub mounts your MyData at ~/MyData; scratch goes there
# since community folders are read-only. Anywhere else, work beside the
# notebook; local folders upload automatically at submission.
mydata = Path.home() / "MyData"
work_root = mydata if mydata.is_dir() else Path.cwd()

## The analysis script (given)

One OpenSeesPy pushover per task. PyLauncher passes each task its `--NodalMass`, `--LCol`, and `--outDir`, and the recorders write into that task's own output folder.

In [ ]:
input_dir = work_root / "opensees_sweep"
input_dir.mkdir(parents=True, exist_ok=True)

cantilever_script = """\
# Ex1a.Canti2D.Push — OpenSeesPy cantilever pushover
# Based on the OpenSees Examples Manual
# Units: kip, inch, second
#
# Command-line arguments (set by PyLauncher per task):
#   --NodalMass  mass at free node
#   --LCol       column length
#   --outDir     output directory for this run

import argparse
import os

if os.path.exists("opensees.so"):
    import opensees as ops
else:
    import openseespy.opensees as ops

parser = argparse.ArgumentParser()
parser.add_argument("--NodalMass", type=float, required=True)
parser.add_argument("--LCol", type=float, required=True)
parser.add_argument("--outDir", type=str, required=True)
args = parser.parse_args()

NodalMass = args.NodalMass
LCol = args.LCol
outDir = args.outDir

os.makedirs(outDir, exist_ok=True)
print(f"Running: NodalMass={NodalMass}, LCol={LCol}, outDir={outDir}")

ops.wipe()
ops.model("basic", "-ndm", 2, "-ndf", 3)

# Geometry
ops.node(1, 0, 0)
ops.node(2, 0, LCol)
ops.fix(1, 1, 1, 1)
ops.mass(2, NodalMass, 0.0, 0.0)

# Element
ops.geomTransf("Linear", 1)
ops.element("elasticBeamColumn", 1, 1, 2, 3600000000, 4227, 1080000, 1)

# Recorders
ops.recorder("Node", "-file", f"{outDir}/DFree.out", "-time", "-node", 2, "-dof", 1, 2, 3, "disp")
ops.recorder("Node", "-file", f"{outDir}/RBase.out", "-time", "-node", 1, "-dof", 1, 2, 3, "reaction")
ops.recorder("Element", "-file", f"{outDir}/FCol.out", "-time", "-ele", 1, "globalForce")

# Gravity analysis
ops.timeSeries("Linear", 1)
ops.pattern("Plain", 1, 1)
ops.load(2, 0.0, -2000.0, 0.0)
ops.wipeAnalysis()
ops.constraints("Plain")
ops.numberer("Plain")
ops.system("BandGeneral")
ops.test("NormDispIncr", 1.0e-8, 6)
ops.algorithm("Newton")
ops.integrator("LoadControl", 0.1)
ops.analysis("Static")
ops.analyze(10)
ops.loadConst("-time", 0.0)

# Pushover analysis
ops.timeSeries("Linear", 2)
ops.pattern("Plain", 2, 2)
ops.load(2, 2000.0, 0.0, 0.0)
ops.integrator("DisplacementControl", 2, 1, 0.1)
ops.analyze(1000)

print(f"Done: NodalMass={NodalMass}, LCol={LCol}")
"""

(input_dir / "cantilever.py").write_text(cantilever_script)
print(f"Wrote {input_dir}/cantilever.py")

## Stage OpenSeesPy

`python-s3` loads no modules beyond Python by default. The tasks import the TACC-compiled OpenSeesPy, so a pre-script copies it into the job directory — `EXTRA_MODULES=opensees,hdf5/1.14.4` (passed at submit) provides `TACC_OPENSEES_BIN` and the runtime libraries.

In [ ]:
setup_script = """\
#!/bin/bash
# Pre-script: stage the TACC-compiled OpenSeesPy next to the tasks.
# The bundled filename differs across opensees module versions.
for f in "${TACC_OPENSEES_BIN}/opensees.so" "${TACC_OPENSEES_BIN}/OpenSeesPy.so"; do
    [ -f "$f" ] && cp "$f" ./opensees.so && exit 0
done
echo "ERROR: no OpenSeesPy library in ${TACC_OPENSEES_BIN}" >&2; exit 1
"""

(input_dir / "setup.sh").write_text(setup_script)
print("Wrote setup.sh")

## TODO 1. Define the sweep

Sweep the nodal mass over five values from 4.19 to 4.99 and the column length over 100, 200, and 300. Every combination becomes one task.

<details><summary>Hint</summary>

A dict of lists. The placeholder names must match the `--NodalMass` and `--LCol` tokens in the command template, `NODAL_MASS` and `LCOL`.
</details>

Docs: [PyLauncher Parameter Sweeps](https://designsafe-ci.github.io/dapi/examples/pylauncher).

In [ ]:
sweep = ...  # TODO

command = (
    "python3 cantilever.py --NodalMass {NODAL_MASS} --LCol {LCOL} "
    "--outDir out_{NODAL_MASS}_{LCOL}"
)

## TODO 2. Preview before generating

`generate()` with `preview=True` shows every command the sweep would run, without writing anything. Count the tasks; five masses times three lengths should give fifteen.

<details><summary>Hint</summary>

`ds.jobs.parametric_sweep.generate(command, sweep, str(input_dir), preview=True)` returns a DataFrame of the expanded commands.
</details>

In [ ]:
df = ...  # TODO
df

## TODO 3. Generate and submit

Generate the sweep files for real, then submit the folder as one `python-s3` job on 48 cores of one `skx-dev` node. The job needs the OpenSees module and the `setup.sh` pre-script that stages OpenSeesPy.

<details><summary>Hint</summary>

`generate(command, sweep, str(input_dir))` writes the PyLauncher inputs. Then `ds.jobs.parametric_sweep.submit(str(input_dir), app_id="python-s3", allocation=..., node_count=1, cores_per_node=48, max_minutes=30, queue="skx-dev", extra_env_vars=[{"key": "EXTRA_MODULES", "value": "opensees,hdf5/1.14.4"}, {"key": "PRE_SCRIPT", "value": "setup.sh"}])`, and `job.monitor(interval=30)` waits it out. A local folder uploads automatically at submission.
</details>

Docs: [Job Monitoring](https://designsafe-ci.github.io/dapi/jobs#job-monitoring).

In [ ]:
allocation = "DS-Portal-SPARC2026"  # <-- replace with your allocation

commands = ...  # TODO: generate the sweep files
job = ...  # TODO: submit
final = ...  # TODO: monitor to completion
job.print_runtime_summary()

## TODO 4. Plot the pushover curves

Each task archived its recorders into its own `out_<mass>_<length>` folder under the job's `inputDirectory`. For the middle mass (4.59), download `DFree.out` and `RBase.out` for each column length and plot base shear against tip displacement, one curve per length. Which way does stiffness move as the column lengthens?

<details><summary>Hint</summary>

`job.archive_uri + "/inputDirectory"` lists the `out_*` folders. Displacement is column 1 of `DFree.out`, base shear is the negative of column 1 of `RBase.out`.
</details>

Docs: [Output Management](https://designsafe-ci.github.io/dapi/jobs#output-management).

In [ ]:
import numpy as np  # noqa: F401  (your answer uses it)
import matplotlib.pyplot as plt  # noqa: F401  (your answer uses it)

%matplotlib inline

results_uri = ...  # TODO
# TODO: download and plot one curve per column length

## Going further

Add a third sweep parameter, the pushover load, and watch the task count multiply. The [PyLauncher sweep example](pylauncher_sweep.ipynb) covers the mechanics, and the [OpenSees ML workflow](../opensees_ml/DS_OpenSees_ML_Workflow_DAG.ipynb) feeds a sweep like this one into a training job.